In [1]:
import pandas as pd
import glob

files = glob.glob("sap-o2c-data/billing_document_headers/*.jsonl")

df_list = [pd.read_json(f, lines=True) for f in files]

df = pd.concat(df_list, ignore_index=True)

print(df.shape)
print(df.head())

(163, 14)
   billingDocument billingDocumentType              creationDate  \
0         90504248                  F2  2025-04-03T00:00:00.000Z   
1         90628265                  F2  2025-05-09T00:00:00.000Z   
2         90628266                  F2  2025-05-09T00:00:00.000Z   
3         90504274                  F2  2025-04-03T00:00:00.000Z   
4         90504242                  F2  2025-04-03T00:00:00.000Z   

                                  creationTime        lastChangeDateTime  \
0  {'hours': 11, 'minutes': 31, 'seconds': 13}  2025-04-03T11:31:37.331Z   
1   {'hours': 7, 'minutes': 21, 'seconds': 19}  2025-05-09T07:21:26.332Z   
2   {'hours': 7, 'minutes': 21, 'seconds': 19}  2025-05-09T07:21:26.695Z   
3  {'hours': 11, 'minutes': 31, 'seconds': 13}  2025-07-24T11:42:30.485Z   
4  {'hours': 11, 'minutes': 31, 'seconds': 13}  2025-07-24T11:42:32.739Z   

        billingDocumentDate  billingDocumentIsCancelled  \
0  2025-04-02T00:00:00.000Z                       False   
1  202

In [2]:
print(list(df.columns))

['billingDocument', 'billingDocumentType', 'creationDate', 'creationTime', 'lastChangeDateTime', 'billingDocumentDate', 'billingDocumentIsCancelled', 'cancelledBillingDocument', 'totalNetAmount', 'transactionCurrency', 'companyCode', 'fiscalYear', 'accountingDocument', 'soldToParty']


In [3]:
df['creationTime_hours'] = df['creationTime'].apply(lambda x: x.get('hours') if isinstance(x, dict) else None)
df['creationTime_minutes'] = df['creationTime'].apply(lambda x: x.get('minutes') if isinstance(x, dict) else None)
df['creationTime_seconds'] = df['creationTime'].apply(lambda x: x.get('seconds') if isinstance(x, dict) else None)

df.drop(columns=['creationTime'], inplace=True)

In [4]:
df = df.rename(columns={
    "billingDocument": "invoice_id",
    "soldToParty": "customer_id",
    "billingDocumentDate": "invoice_date",
    "totalNetAmount": "amount",
    "transactionCurrency": "currency",
    "billingDocumentIsCancelled": "is_cancelled",
    "accountingDocument": "accounting_id"
})

In [5]:
df = df[[
    "invoice_id",
    "customer_id",
    "invoice_date",
    "amount",
    "currency",
    "is_cancelled",
    "accounting_id"
]]

In [6]:
import sqlite3

conn = sqlite3.connect("data.db")
df.to_sql("invoices", conn, if_exists="replace", index=False)

163

In [7]:
df.to_sql("invoices", conn, if_exists="replace", index=False)

163

In [17]:
print(len(df))

163
